# 🐘 Elephant Detection Exploration - MegaDetector Testing (CORRECTED)

## Purpose
Interactive exploration and validation of detection-based preprocessing pipeline.

**Objectives:**
1. Test MegaDetector on elephant re-ID dataset
2. Validate arrow-to-box matching logic
3. Tune confidence threshold and padding ratio
4. Visualize detection quality
5. Determine optimal parameters for production pipeline

---

## 1. Setup & Dependencies

In [1]:
import os
import sys
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import json
from typing import List, Tuple, Optional, Dict
import random
from collections import defaultdict

# MegaDetector imports
from megadetector.detection import run_detector
from megadetector.visualization import visualization_utils as vis_utils

# Configure matplotlib for inline display
%matplotlib inline
plt.rcParams['figure.figsize'] = (16, 10)
plt.rcParams['figure.dpi'] = 100

print("✓ Dependencies loaded")

✓ Dependencies loaded


## 2. Configuration & Paths

In [2]:
# Root paths
PROJECT_ROOT = Path("..")
DATA_ROOT = PROJECT_ROOT / "data"

# Dataset paths
MAKHNA_RAW = DATA_ROOT / "raw" / "Makhna_id_udalguri_24" / "Makhna_id_udalguri_24"
HERD_RAW = DATA_ROOT / "raw" / "Herd_ID_Udalguri_24" / "Herd_ID_Udalguri_24"

# Detection parameters
CONFIDENCE_THRESHOLDS = [0.3, 0.4, 0.5, 0.6]  # Test different thresholds
PADDING_RATIOS = [0.10, 0.15, 0.20]  # Test different padding
DEFAULT_CONFIDENCE = 0.4  # Starting point
DEFAULT_PADDING = 0.15

# Arrow detection parameters (from existing preprocessing)
MIN_ARROW_AREA = 4000
ARROW_HSV_LOWER1 = np.array([0, 100, 100])
ARROW_HSV_UPPER1 = np.array([10, 255, 255])
ARROW_HSV_LOWER2 = np.array([160, 100, 100])
ARROW_HSV_UPPER2 = np.array([180, 255, 255])

print(f"✓ Project root: {PROJECT_ROOT.absolute()}")
print(f"✓ Makhna dataset: {MAKHNA_RAW.exists()}")
print(f"✓ Herd dataset: {HERD_RAW.exists()}")

✓ Project root: d:\Elephant_ReIdentification\notebooks\..
✓ Makhna dataset: True
✓ Herd dataset: False


## 3. Load MegaDetector Model

In [3]:
# Load MegaDetector model
print("Loading MegaDetector v5...")
detector = run_detector.load_detector('MDV5A')  # MegaDetector v5a
print("✓ MegaDetector loaded successfully")
print(f"Model type: {type(detector)}")

Loading MegaDetector v5...
Model v5a.0.1 already exists and is valid at C:\Users\giris\AppData\Local\Temp\megadetector_models\md_v5a.0.1.pt
PyTorch reports 1 available CUDA devices
GPU available: True


C:\Users\giris\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\yolov5\utils\general.py:31: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources as pkg


Loading PT detector with compatibility mode classic
Loaded image size 1280 from model metadata
Using model stride: 64
PTDetector using device cuda:0


Fusing layers... 
Fusing layers... 
Model summary: 733 layers, 140054656 parameters, 0 gradients, 208.8 GFLOPs
Model summary: 733 layers, 140054656 parameters, 0 gradients, 208.8 GFLOPs


✓ MegaDetector loaded successfully
Model type: <class 'megadetector.detection.pytorch_detector.PTDetector'>


## 4. Utility Functions

### 4.1 Arrow Detection

In [4]:
def detect_arrow_tip(image: np.ndarray) -> Optional[Tuple[int, int]]:
    """
    Detect red arrow and return tip coordinates.
    
    Args:
        image: BGR image (numpy array)
        
    Returns:
        tuple (x, y): Arrow tip coordinates if valid arrow found
        None: If no arrow or invalid arrow
    """
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    
    # Two HSV ranges for red (wraps around 180)
    mask1 = cv2.inRange(hsv, ARROW_HSV_LOWER1, ARROW_HSV_UPPER1)
    mask2 = cv2.inRange(hsv, ARROW_HSV_LOWER2, ARROW_HSV_UPPER2)
    mask = cv2.bitwise_or(mask1, mask2)
    
    # Morphological cleanup
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)
    
    # Find largest contour
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    if not contours:
        return None
    
    # Get largest arrow candidate
    arrow_contour = max(contours, key=cv2.contourArea)
    area = cv2.contourArea(arrow_contour)
    
    if area < MIN_ARROW_AREA:
        return None
    
    # Find topmost point (arrow tip points DOWN, so we want highest y)
    moments = cv2.moments(arrow_contour)
    if moments['m00'] == 0:
        return None
    
    cx = int(moments['m10'] / moments['m00'])
    cy = int(moments['m01'] / moments['m00'])
    
    return (cx, cy)

print("✓ Arrow detection function defined")

✓ Arrow detection function defined


### 4.2 Elephant Detection (✅ CORRECTED API)

In [5]:
def detect_elephants(image_path: str, confidence_threshold: float = 0.4) -> List[Dict]:
    """
    Detect elephants in image using MegaDetector.
    
    Args:
        image_path: Path to image file
        confidence_threshold: Minimum confidence for detection
        
    Returns:
        List of detection dicts with keys: bbox, conf, category
    """
    # Load image using MegaDetector's utility
    image = vis_utils.load_image(image_path)
    if image is None:
        return []
    
    # ✅ CORRECT: Use generate_detections_one_image method
    result = detector.generate_detections_one_image(image)
    
    if not result or 'detections' not in result:
        return []
    
    detections = result['detections']
    
    # Filter for animal category (category '1') and above threshold
    elephant_detections = [
        det for det in detections 
        if det.get('category', '') == '1' and det.get('conf', 0) >= confidence_threshold
    ]
    
    return elephant_detections

print("✓ Elephant detection function defined (CORRECTED)")

✓ Elephant detection function defined (CORRECTED)


### 4.3 Arrow-to-Box Matching

In [6]:
def point_in_box(point: Tuple[int, int], bbox: List[float], img_width: int, img_height: int) -> bool:
    """
    Check if point is inside bounding box.
    
    Args:
        point: (x, y) coordinates
        bbox: [x_norm, y_norm, w_norm, h_norm] in normalized coordinates
        img_width, img_height: Image dimensions for denormalization
    """
    px, py = point
    
    # Convert normalized bbox to pixel coordinates
    x_norm, y_norm, w_norm, h_norm = bbox
    x1 = int(x_norm * img_width)
    y1 = int(y_norm * img_height)
    x2 = int((x_norm + w_norm) * img_width)
    y2 = int((y_norm + h_norm) * img_height)
    
    return x1 <= px <= x2 and y1 <= py <= y2

def box_area(bbox: List[float]) -> float:
    """Calculate box area (normalized coordinates)."""
    return bbox[2] * bbox[3]

def distance_to_box(point: Tuple[int, int], bbox: List[float], img_width: int, img_height: int) -> float:
    """Calculate distance from point to box center."""
    px, py = point
    x_norm, y_norm, w_norm, h_norm = bbox
    
    # Box center in pixels
    cx = (x_norm + w_norm/2) * img_width
    cy = (y_norm + h_norm/2) * img_height
    
    return np.sqrt((px - cx)**2 + (py - cy)**2)

def select_target_elephant(
    detections: List[Dict], 
    arrow_tip: Optional[Tuple[int, int]], 
    img_width: int, 
    img_height: int
) -> Optional[Dict]:
    """
    Select target elephant based on arrow or size.
    
    Logic:
    - If arrow present: return box containing arrow tip
    - If no arrow: return largest box
    - If arrow but no match: return closest box to arrow
    """
    if not detections:
        return None
    
    if arrow_tip is not None:
        # Try to find box containing arrow
        for det in detections:
            bbox = det['bbox']
            if point_in_box(arrow_tip, bbox, img_width, img_height):
                return det
        
        # Fallback: closest box to arrow
        closest = min(detections, key=lambda d: distance_to_box(arrow_tip, d['bbox'], img_width, img_height))
        return closest
    else:
        # No arrow: largest box
        largest = max(detections, key=lambda d: box_area(d['bbox']))
        return largest

def check_ambiguity(detections: List[Dict]) -> bool:
    """Check if multiple similar-sized elephants detected."""
    if len(detections) <= 1:
        return False
    
    areas = [box_area(det['bbox']) for det in detections]
    max_area = max(areas)
    similar_boxes = [a for a in areas if a >= 0.85 * max_area]
    
    return len(similar_boxes) > 1

print("✓ Matching functions defined")

✓ Matching functions defined


### 4.4 Visualization

In [7]:
def visualize_detection_result(
    image: np.ndarray,
    detections: List[Dict],
    arrow_tip: Optional[Tuple[int, int]],
    selected_detection: Optional[Dict],
    image_path: str
):
    """
    Visualize detection results with bounding boxes and arrow.
    """
    img_display = image.copy()
    h, w = img_display.shape[:2]
    
    # Draw all detections in yellow
    for det in detections:
        bbox = det['bbox']
        conf = det.get('conf', 0)
        
        x1 = int(bbox[0] * w)
        y1 = int(bbox[1] * h)
        x2 = int((bbox[0] + bbox[2]) * w)
        y2 = int((bbox[1] + bbox[3]) * h)
        
        cv2.rectangle(img_display, (x1, y1), (x2, y2), (0, 255, 255), 2)
        cv2.putText(img_display, f"{conf:.2f}", (x1, y1-5), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)
    
    # Draw selected detection in green
    if selected_detection:
        bbox = selected_detection['bbox']
        x1 = int(bbox[0] * w)
        y1 = int(bbox[1] * h)
        x2 = int((bbox[0] + bbox[2]) * w)
        y2 = int((bbox[1] + bbox[3]) * h)
        
        cv2.rectangle(img_display, (x1, y1), (x2, y2), (0, 255, 0), 4)
        cv2.putText(img_display, "SELECTED", (x1, y1-25), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
    
    # Draw arrow tip in red
    if arrow_tip:
        cv2.circle(img_display, arrow_tip, 10, (0, 0, 255), -1)
        cv2.putText(img_display, "ARROW", (arrow_tip[0]+15, arrow_tip[1]), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
    
    # Display
    plt.figure(figsize=(18, 12))
    plt.imshow(cv2.cvtColor(img_display, cv2.COLOR_BGR2RGB))
    plt.title(f"{Path(image_path).name} | Detections: {len(detections)} | Arrow: {arrow_tip is not None}", 
             fontsize=14)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

def visualize_crop_comparison(
    image: np.ndarray,
    selected_detection: Dict,
    padding_ratios: List[float]
):
    """
    Compare crops with different padding ratios.
    """
    h, w = image.shape[:2]
    bbox = selected_detection['bbox']
    
    fig, axes = plt.subplots(1, len(padding_ratios), figsize=(18, 6))
    if len(padding_ratios) == 1:
        axes = [axes]
    
    for ax, padding in zip(axes, padding_ratios):
        # Extract crop with padding
        x1 = int(bbox[0] * w)
        y1 = int(bbox[1] * h)
        x2 = int((bbox[0] + bbox[2]) * w)
        y2 = int((bbox[1] + bbox[3]) * h)
        
        # Add padding
        box_w = x2 - x1
        box_h = y2 - y1
        pad_w = int(box_w * padding)
        pad_h = int(box_h * padding)
        
        x1_pad = max(0, x1 - pad_w)
        y1_pad = max(0, y1 - pad_h)
        x2_pad = min(w, x2 + pad_w)
        y2_pad = min(h, y2 + pad_h)
        
        crop = image[y1_pad:y2_pad, x1_pad:x2_pad]
        
        ax.imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
        ax.set_title(f"Padding: {padding:.0%}", fontsize=12)
        ax.axis('off')
    
    plt.tight_layout()
    plt.show()

print("✓ Visualization functions defined")

✓ Visualization functions defined


## 5. Sample Image Selection

In [8]:
def collect_sample_images(num_samples: int = 10) -> List[str]:
    """
    Collect random sample images from both datasets.
    """
    samples = []
    valid_extensions = {'.jpg', '.jpeg', '.JPG', '.JPEG', '.png', '.PNG'}
    
    # Collect from Makhna dataset
    makhna_images = []
    for root, dirs, files in os.walk(MAKHNA_RAW):
        for file in files:
            if Path(file).suffix in valid_extensions:
                makhna_images.append(os.path.join(root, file))
    
    # Collect from Herd dataset
    herd_images = []
    for root, dirs, files in os.walk(HERD_RAW):
        for file in files:
            if Path(file).suffix in valid_extensions:
                herd_images.append(os.path.join(root, file))
    
    # Random sampling
    if makhna_images:
        samples.extend(random.sample(makhna_images, min(num_samples//2, len(makhna_images))))
    if herd_images:
        samples.extend(random.sample(herd_images, min(num_samples//2, len(herd_images))))
    
    return samples

# Collect sample images
SAMPLE_IMAGES = collect_sample_images(num_samples=20)
print(f"✓ Collected {len(SAMPLE_IMAGES)} sample images")
for i, img_path in enumerate(SAMPLE_IMAGES[:5], 1):
    print(f"  {i}. {Path(img_path).name}")
print("  ...")

✓ Collected 10 sample images
  1. DSCN5614.JPG
  2. DSCN5128.JPG
  3. DSCN5115.JPG
  4. DSCN4809.JPG
  5. DSCN4270.JPG
  ...


## 6. Detection Testing on Sample Images

Test MegaDetector on sample images and visualize results.

In [9]:
# Test detection on first 5 samples
test_results = []

for img_path in SAMPLE_IMAGES[:5]:
    print(f"\n{'='*80}")
    print(f"Processing: {Path(img_path).name}")
    print(f"{'='*80}")
    
    # Load image for arrow detection
    image = cv2.imread(img_path)
    if image is None:
        print(f"❌ Failed to load image")
        continue
    
    h, w = image.shape[:2]
    print(f"Image size: {w} × {h}")
    
    # Detect arrow
    arrow_tip = detect_arrow_tip(image)
    print(f"Arrow detected: {arrow_tip is not None}")
    if arrow_tip:
        print(f"  Arrow tip at: {arrow_tip}")
    
    # Detect elephants
    detections = detect_elephants(img_path, confidence_threshold=DEFAULT_CONFIDENCE)
    print(f"Elephants detected: {len(detections)}")
    for i, det in enumerate(detections, 1):
        print(f"  {i}. Confidence: {det.get('conf', 0):.3f}, BBox: {det['bbox']}")
    
    # Select target elephant
    selected = select_target_elephant(detections, arrow_tip, w, h)
    if selected:
        print(f"✓ Selected elephant with confidence: {selected.get('conf', 0):.3f}")
    else:
        print(f"❌ No elephant selected")
    
    # Check ambiguity
    if check_ambiguity(detections):
        print(f"⚠️  AMBIGUOUS: Multiple similar-sized elephants detected")
    
    # Visualize
    visualize_detection_result(image, detections, arrow_tip, selected, img_path)
    
    # Store results
    test_results.append({
        'image': Path(img_path).name,
        'arrow_present': arrow_tip is not None,
        'num_detections': len(detections),
        'selected_confidence': selected.get('conf', 0) if selected else None,
        'ambiguous': check_ambiguity(detections)
    })

print(f"\n✓ Tested {len(test_results)} images")


Processing: DSCN5614.JPG


Image size: 4608 × 3456
Arrow detected: False
Elephants detected: 1
  1. Confidence: 0.723, BBox: [0.0648, 0.2737, 0.7389, 0.7262]
✓ Selected elephant with confidence: 0.723


C:\Users\giris\AppData\Local\Temp\ipykernel_10184\2714530703.py:53: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



Processing: DSCN5128.JPG
Image size: 4608 × 3456
Arrow detected: False
Elephants detected: 1
  1. Confidence: 0.972, BBox: [0.2211, 0.0905, 0.3736, 0.7552]
✓ Selected elephant with confidence: 0.972


C:\Users\giris\AppData\Local\Temp\ipykernel_10184\2714530703.py:53: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


...

C:\Users\giris\AppData\Local\Temp\ipykernel_10184\2714530703.py:53: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Image size: 4608 × 3456
Arrow detected: False
Elephants detected: 1
  1. Confidence: 0.973, BBox: [0.1742, 0.0888, 0.7131, 0.8709]
✓ Selected elephant with confidence: 0.973

Processing: DSCN4270.JPG


C:\Users\giris\AppData\Local\Temp\ipykernel_10184\2714530703.py:53: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Image size: 4608 × 3456
Arrow detected: True
  Arrow tip at: (2173, 857)
Elephants detected: 3
  1. Confidence: 0.490, BBox: [0.2402, 0.7329, 0.0937, 0.0943]
  2. Confidence: 0.949, BBox: [0.0004, 0.5379, 0.1549, 0.2774]
  3. Confidence: 0.955, BBox: [0.1534, 0.3313, 0.5625, 0.4953]
✓ Selected elephant with confidence: 0.955

✓ Tested 5 images


C:\Users\giris\AppData\Local\Temp\ipykernel_10184\2714530703.py:53: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Confidence Threshold Tuning

Test different confidence thresholds to find optimal value.

In [10]:
# Collect detection confidences across different thresholds
threshold_stats = defaultdict(list)

for img_path in SAMPLE_IMAGES[:10]:
    for threshold in CONFIDENCE_THRESHOLDS:
        detections = detect_elephants(img_path, confidence_threshold=threshold)
        threshold_stats[threshold].append(len(detections))

# Visualize threshold impact
plt.figure(figsize=(12, 6))
for threshold in CONFIDENCE_THRESHOLDS:
    counts = threshold_stats[threshold]
    plt.plot(range(len(counts)), counts, marker='o', label=f"Threshold {threshold}")

plt.xlabel("Image Index", fontsize=12)
plt.ylabel("Number of Detections", fontsize=12)
plt.title("Detection Count vs Confidence Threshold", fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Summary statistics
print("\nThreshold Statistics:")
print("=" * 60)
for threshold in CONFIDENCE_THRESHOLDS:
    counts = threshold_stats[threshold]
    avg_detections = np.mean(counts)
    zero_detections = sum(1 for c in counts if c == 0)
    print(f"Threshold {threshold}:")
    print(f"  Avg detections: {avg_detections:.2f}")
    print(f"  Images with 0 detections: {zero_detections}/{len(counts)}")
    print()


Threshold Statistics:
Threshold 0.3:
  Avg detections: 1.60
  Images with 0 detections: 0/10

Threshold 0.4:
  Avg detections: 1.50
  Images with 0 detections: 0/10

Threshold 0.5:
  Avg detections: 1.30
  Images with 0 detections: 0/10

Threshold 0.6:
  Avg detections: 1.30
  Images with 0 detections: 0/10



C:\Users\giris\AppData\Local\Temp\ipykernel_10184\3996892002.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Padding Ratio Comparison

Compare crop quality with different padding ratios.

In [11]:
# Test padding on images with successful detection
for img_path in SAMPLE_IMAGES[:3]:
    print(f"\nTesting padding on: {Path(img_path).name}")
    
    image = cv2.imread(img_path)
    if image is None:
        continue
    
    h, w = image.shape[:2]
    detections = detect_elephants(img_path, confidence_threshold=DEFAULT_CONFIDENCE)
    arrow_tip = detect_arrow_tip(image)
    selected = select_target_elephant(detections, arrow_tip, w, h)
    
    if selected:
        visualize_crop_comparison(image, selected, PADDING_RATIOS)
    else:
        print("  No detection - skipping")


Testing padding on: DSCN5614.JPG


C:\Users\giris\AppData\Local\Temp\ipykernel_10184\2714530703.py:95: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



Testing padding on: DSCN5128.JPG


C:\Users\giris\AppData\Local\Temp\ipykernel_10184\2714530703.py:95: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



Testing padding on: DSCN5115.JPG


C:\Users\giris\AppData\Local\Temp\ipykernel_10184\2714530703.py:95: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9. Failure Analysis

Identify and analyze cases where detection fails.

In [12]:
# Test on all samples and collect failures
failures = []
successes = []

for img_path in SAMPLE_IMAGES:
    detections = detect_elephants(img_path, confidence_threshold=DEFAULT_CONFIDENCE)
    
    if len(detections) == 0:
        failures.append(img_path)
    else:
        successes.append(img_path)

detection_rate = len(successes) / len(SAMPLE_IMAGES) * 100

print(f"\nDetection Results:")
print(f"=" * 60)
print(f"Total samples: {len(SAMPLE_IMAGES)}")
print(f"Successful detections: {len(successes)} ({detection_rate:.1f}%)")
print(f"Failed detections: {len(failures)} ({100-detection_rate:.1f}%)")

if failures:
    print(f"\nFailed images:")
    for fail_path in failures:
        print(f"  - {Path(fail_path).name}")


Detection Results:
Total samples: 10
Successful detections: 10 (100.0%)
Failed detections: 0 (0.0%)


## 10. Summary & Recommendations

In [13]:
print("\n" + "=" * 80)
print("EXPLORATION SUMMARY")
print("=" * 80)

print("\n📊 Key Findings:")
print(f"  • Detection rate: {detection_rate:.1f}%")
print(f"  • Arrow detection: {sum(1 for r in test_results if r['arrow_present'])}/{len(test_results)} images")
print(f"  • Ambiguous cases: {sum(1 for r in test_results if r['ambiguous'])}/{len(test_results)} images")

print("\n🎯 Recommended Parameters:")
print(f"  • Confidence threshold: {DEFAULT_CONFIDENCE} (adjust based on threshold analysis above)")
print(f"  • Padding ratio: {DEFAULT_PADDING} (adjust based on crop comparison above)")

print("\n✅ Next Steps:")
print("  1. Review visualizations and adjust parameters if needed")
print("  2. Extract validated code to production modules (detector.py, etc.)")
print("  3. Run full dataset preprocessing with tuned parameters")
print("  4. Review rejected/ folder for failed detections")

print("\n" + "=" * 80)


EXPLORATION SUMMARY

📊 Key Findings:
  • Detection rate: 100.0%
  • Arrow detection: 1/5 images
  • Ambiguous cases: 0/5 images

🎯 Recommended Parameters:
  • Confidence threshold: 0.4 (adjust based on threshold analysis above)
  • Padding ratio: 0.15 (adjust based on crop comparison above)

✅ Next Steps:
  1. Review visualizations and adjust parameters if needed
  2. Extract validated code to production modules (detector.py, etc.)
  3. Run full dataset preprocessing with tuned parameters
  4. Review rejected/ folder for failed detections



## Notes & Observations

*Add your observations here as you explore:*

- Detection quality:
- Threshold findings:
- Padding findings:
- Edge cases:
- Recommendations: